# TP1 - Import de formats et accès aux API
## Des petits scripts python pour nous aider

In [2]:
from pathlib import Path
import pandas as pd
import geopandas as gpd

### 0 - Lire un jeu de donnée au format CSV avec geopandas

In [3]:
dossier_courant = Path.cwd()
dossier_donnees = dossier_courant / "data"
dossier_sortie = dossier_donnees / "output"
dossier_sortie.mkdir(parents=True, exist_ok=True)

In [4]:
print(f"Dossier de travail actuel : {dossier_courant}")

Dossier de travail actuel : c:\Users\user\Documents\tp-data\tp1


In [5]:
fichier_csv = dossier_donnees / "human_chamois_encounters.csv"
colonne_wkt = "eca_h"
crs_source = "EPSG:2154"

In [6]:
df = pd.read_csv(fichier_csv)
geometry = gpd.GeoSeries.from_wkt(df[colonne_wkt], crs=crs_source)

if colonne_wkt != "geometry":
    df = df.drop(columns=[colonne_wkt])

gdf = gpd.GeoDataFrame(df, geometry=geometry)

In [6]:
gdf.head()

,id_encounter,type,id_indiv_human,id_indiv_animal,id_traj_human,id_traj_animal,date_human,date_animal,earliest_human,latest_human,earliest_animal,latest_animal,eca_a,geometry
0,1,Potential,19934,1,20533,20,2020-06-27,2013-07-09,06:21:08,06:23:08,06:20:34,06:30:41,"POLYGON((953125.181568785 6508201.833860811,95...","POLYGON ((952977.158 6508367.925, 952897.225 6..."
1,2,Potential,19220,1,20571,20,2020-07-07,2013-07-09,13:52:44,13:53:47,13:50:51,14:00:26,"POLYGON((953051.7089725762 6508157.022283436,9...","POLYGON ((952878.158 6508338.26, 952876.251 65..."
2,3,Potential,21741,1,21741,20,2020-06-23,2013-07-09,13:44:14,13:45:30,13:40:47,13:50:51,"POLYGON((953089.0731599241 6508149.807691646,9...","POLYGON ((952893.81 6508342.013, 952891.886 65..."
3,4,Potential,19376,21,19558,21,2018-08-12,2012-08-05,15:16:23,15:34:35,15:00:48,16:00:47,"POLYGON((948739.8833260989 6514643.379612487,9...","POLYGON ((948499.153 6514567.832, 948498.945 6..."
4,5,Potential,19193,21,19590,21,2012-07-22,2012-08-05,11:28:59,11:41:41,11:00:31,12:00:53,"POLYGON((948692.9600541465 6514687.195881412,9...","POLYGON ((948518.354 6514642.793, 948518.436 6..."


In [7]:
print("Lignes, colonnes :", gdf.shape)
print("CRS :", gdf.crs)
print("Emprise :", gdf.total_bounds)
print("\nTypes de géométries :", gdf.geom_type.value_counts())
print("\nGéométries invalides :", (~gdf.is_valid).sum())

Lignes, colonnes : (63759, 14)
CRS : EPSG:2154
Emprise : [ 947550.02448383 6505444.23466601  955135.96371892 6517307.34002861]

Types de géométries : Polygon         45353
MultiPolygon    18406
Name: count, dtype: int64

Géométries invalides : 0


### 1 - Convertir vers un autre format de données

In [8]:
nom_base = fichier_csv.stem

In [9]:
gdf.to_file(dossier_sortie / f"{nom_base}.geojson", driver="GeoJSON")
gdf.to_file(dossier_sortie / f"{nom_base}.shp", driver="ESRI Shapefile")
gdf.to_file(dossier_sortie / f"{nom_base}.gpkg", driver="GPKG")
gdf.to_parquet(dossier_sortie / f"{nom_base}.parquet")

C:\Users\user\AppData\Local\Temp\ipykernel_9576\1858458494.py:2: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(dossier_sortie / f"{nom_base}.shp", driver="ESRI Shapefile")
c:\Users\user\Documents\tp-data\tp1\.venv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'id_encounter' to 'id_encount'
  ogr_write(
c:\Users\user\Documents\tp-data\tp1\.venv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'id_indiv_human' to 'id_indiv_h'
  ogr_write(
c:\Users\user\Documents\tp-data\tp1\.venv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'id_indiv_animal' to 'id_indiv_a'
  ogr_write(
c:\Users\user\Documents\tp-data\tp1\.venv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'id_traj_human' to 'id_traj_hu'
  ogr_write(
c:\Users\user\Documents\tp-data\tp1\.venv\Lib\site-packages\py

### 2 - Mesures et analyses des fichiers

In [10]:
import time

In [11]:
dossier_sortie_mesures = dossier_sortie / "mesures"
dossier_sortie_mesures.mkdir(parents=True, exist_ok=True)

In [12]:
formats = {
    "Shapefile": (dossier_sortie_mesures / f"{nom_base}.shp", "ESRI Shapefile"),
    "GeoJSON": (dossier_sortie_mesures / f"{nom_base}.geojson", "GeoJSON"),
    "GeoPackage": (dossier_sortie_mesures / f"{nom_base}.gpkg", "GPKG"),
}

In [13]:
rows = []
MB_WINDOWS = 1024 ** 2  # et non 1e6 pour des Mo
# l'explorateur windows appelle Mo ce qui est en réalité des Mio (Mébioctets) soit 1024*1024 octets
exts_shp = {".shp", ".shx", ".dbf", ".prj", ".cpg", ".sbn", ".sbx"}

for nom, (path, driver) in formats.items():
    t0 = time.perf_counter()
    gdf.to_file(path, driver=driver)
    t_write = time.perf_counter() - t0

    t0 = time.perf_counter()
    _ = gpd.read_file(path)
    t_read = time.perf_counter() - t0
    
    if nom == "Shapefile":
        size_mb = sum(f.stat().st_size for f in dossier_sortie_mesures.glob(f"{nom_base}.*") if f.suffix.lower() in exts_shp) / MB_WINDOWS
    else:
        size_mb = path.stat().st_size / MB_WINDOWS

    rows.append({
        "Format": nom,
        "Taille (Mo)": round(size_mb, 2),
        "Écriture (s)": round(t_write, 3),
        "Lecture (s)": round(t_read, 3),
    })

parquet_path = dossier_sortie_mesures / f"{nom_base}.parquet"
t0 = time.perf_counter()
gdf.to_parquet(parquet_path, index=False)
t_write = time.perf_counter() - t0

t0 = time.perf_counter()
_ = gpd.read_parquet(parquet_path)
t_read = time.perf_counter() - t0

rows.append({
    "Format": "GeoParquet",
    "Taille (Mo)": round(parquet_path.stat().st_size / MB_WINDOWS, 2),
    "Écriture (s)": round(t_write, 3),
    "Lecture (s)": round(t_read, 3),
})

C:\Users\user\AppData\Local\Temp\ipykernel_9576\1899756707.py:8: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(path, driver=driver)
c:\Users\user\Documents\tp-data\tp1\.venv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'id_encounter' to 'id_encount'
  ogr_write(
c:\Users\user\Documents\tp-data\tp1\.venv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'id_indiv_human' to 'id_indiv_h'
  ogr_write(
c:\Users\user\Documents\tp-data\tp1\.venv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'id_indiv_animal' to 'id_indiv_a'
  ogr_write(
c:\Users\user\Documents\tp-data\tp1\.venv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'id_traj_human' to 'id_traj_hu'
  ogr_write(
c:\Users\user\Documents\tp-data\tp1\.venv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normal

In [14]:
print(pd.DataFrame(rows).to_string(index=False))

    Format  Taille (Mo)  Écriture (s)  Lecture (s)
 Shapefile       199.84         6.133        2.872
   GeoJSON       553.18        52.337       27.606
GeoPackage       271.02         4.832        2.520
GeoParquet       140.92         2.681        1.495


In [ ]:
import time
import numpy as np

N_RUNS = 3
MB_WINDOWS = 1024 ** 2
exts_shp = {".shp", ".shx", ".dbf", ".prj", ".cpg", ".sbn", ".sbx"}

tous_formats = {
    **formats,
    "GeoParquet": (dossier_sortie_mesures / f"{nom_base}.parquet", "Parquet")
}

rows2 = []

for nom, (path, driver) in tous_formats.items():
    temps_ecriture = []
    temps_lecture = []

    for _ in range(N_RUNS):
        for f in dossier_sortie_mesures.glob(f"{nom_base}.*"):
            f.unlink()

        # écriture
        t0 = time.perf_counter()
        if driver == "Parquet":
            gdf.to_parquet(path, index=False)
        else:
            gdf.to_file(path, driver=driver)
        temps_ecriture.append(time.perf_counter() - t0)

        # lecture
        t0 = time.perf_counter()
        if driver == "Parquet":
            _ = gpd.read_parquet(path)
        else:
            _ = gpd.read_file(path)
        temps_lecture.append(time.perf_counter() - t0)

    if nom == "Shapefile":
        size_mb = sum(
            f.stat().st_size for f in dossier_sortie_mesures.glob(f"{nom_base}.*")
            if f.suffix.lower() in exts_shp
        ) / MB_WINDOWS
    else:
        size_mb = path.stat().st_size / MB_WINDOWS

    rows2.append({
        "Format": nom,
        "Taille (Mo)": round(size_mb, 2),
        "Écriture médiane (s)": round(np.median(temps_ecriture), 3),
        "Lecture médiane (s)": round(np.median(temps_lecture), 3),
    })
print(pd.DataFrame(rows2).to_string(index=False))

### 3 - Récupérer des entitées dans OSM

In [1]:
import osmnx as ox

In [9]:
fichier_zone = dossier_donnees / "parc.geojson" # contour de la reserve
fichier_sortie = dossier_donnees / "poi.gpkg"
nom_couche_sortie = "all_poi"

zone = gpd.read_file(fichier_zone)
polygone_wgs84 = zone.to_crs("EPSG:4326").geometry.union_all()

#### Pour plusieurs type de POI

In [10]:
tags_poi = {
    "amenity": ["drinking_water", "parking", "picnic_table"],
    "natural": ["spring", "peak", "saddle"],
    "tourism": ["viewpoint", "picnic_site", "alpine_hut", "wilderness_hut"],
}

In [11]:
poi = ox.features_from_polygon(polygone_wgs84, tags=tags_poi).to_crs("EPSG:2154")

poi_parc = gpd.clip(poi, zone)

cols = [c for c in ["name", "amenity", "natural", "tourism", "geometry"] if c in poi_parc.columns]

print(f"Nombre d'objets récupérés dans le parc : {len(poi_parc)}")
poi_parc.to_file(fichier_sortie, layer=nom_couche_sortie, driver="GPKG")

Nombre d'objets récupérés dans le parc : 60
